# Run 1: Astra, pick and place

GPT-6 Astra sees the camera images and picks the next skill. The skills themselves are code.


# ⚠️ Before you run this

- E-stop within reach.
- Nobody inside the arm's reach.
- Table clear of anything breakable.
- `speed_percent` at 10 for the first run.

The model can pick a wrong skill and perception can be wrong. The checks in
`piper_llm/safety.py` reduce the risk; they do not remove it.


In [ ]:
import os, numpy as np
from dotenv import load_dotenv           # pip install python-dotenv
load_dotenv()                            # reads .env, which is not in git

from piper_llm.arm import Arm
from piper_llm.camera import RealSense, Frame
from piper_llm.kinematics import Kinematics
from piper_llm.safety import Governor
from piper_llm.skills import Runner, PICK_PLACE_SKILLS
from piper_llm.task import PickPlace
from piper_llm.perception import Detector
from piper_llm.config import ArmConfig, SceneConfig
from piper_llm.deciders import Astra
from piper_llm.record import Recorder


In [ ]:
# Dry check: no torque. Read the arm, camera and models first.
cam = RealSense()
rgb, depth = cam.frames()
print("camera ok", rgb.shape, "depth range %.2f-%.2f m" % (depth[depth>0].min(), depth.max()))

frame = Frame(cam)                        # needs calibration.json
kin = Kinematics()                        # needs the Menagerie MJCF
print("kinematics ok")


In [ ]:
# Read-only arm check. The arm does not move in this cell.
arm = Arm(ArmConfig(speed_percent=10))
print("joints", np.round(arm.joints(), 3))
print("gripper", round(arm.gripper(), 2))
print("fault:", arm.status_error())


## Home the arm

The arm moves now. Keep clear.


In [ ]:
arm.home()
print("tool pose", np.round(kin.tool_pose(arm.joints()), 3))


## Run

Astra sees both camera views each step and picks one skill. Every motion goes through the safety governor.


In [ ]:
scene = SceneConfig()
rec = Recorder(cam, arm, "run1")       # colour, depth, joints and decisions to out/recordings
gov = Governor()
runner = Runner(arm=arm, kin=kin, gov=gov, scene=scene)
detector = Detector(backend="dino")   # object positions only; Astra picks the skill
task = PickPlace(scene, detector, frame, runner)
astra = Astra(model="gpt-6-astra", effort="high")

INSTRUCTIONS = ("A PiPER arm must pick up the red cube and put it in the tray. "
                "Pick the next skill from what you see and the state; each skill says when it "
                "applies. Approach from above, descend, close, lift, move over the tray, lower, "
                "release, retreat, done.")
# Astra judges the scene from the image: it gets the arm's own state and the heights,
# not the detector's positions. The governor underneath checks the full state.
ASTRA_SEES = ("tool_xyz", "tool_z", "at_grasp_height", "at_carry_height", "at_release_height",
              "at_clear_height", "gripper_closed", "object_held", "object_placed", "fingers_jammed",
              "task_complete")

gov.reset()
for step in range(25):
    rgb, depth = cam.frames()
    pose = kin.tool_pose(arm.joints())
    state = task.observe(rgb, depth, pose, arm)
    if task.problem:
        print("cannot place it:", task.problem)
        rec.note("cannot place it: " + task.problem)
        break
    view = {k: state[k] for k in ASTRA_SEES}
    view["gripper_open"] = round(arm.gripper(), 2)
    view["task"] = "red cube into the tray"
    d = astra.choose(view, PICK_PLACE_SKILLS, INSTRUCTIONS, {"scene": rgb})
    print(step, d.skill, "-", d.note)
    rec.note("%d %s: %s" % (step, d.skill, d.note[:80]))

    ok, why = gov.check_precondition(d.skill, state)
    if not ok:
        print("   refused:", why)
        rec.note("refused: " + why)
        continue
    if d.skill == "done":
        break
    target, yaw, grip = task.target(d.skill, pose)
    if target is None:
        print("   no target for", d.skill)
        continue
    reached, why = runner.move_tool_to(d.skill, target, yaw, grip)
    if why:
        print("   stopped:", why)
        rec.note("stopped: " + why)
    task.after(d.skill, reached, state, pose)

print("safety events:", gov.summary())
print("astra calls:", astra.calls, "median %.0f ms" % (1000*astra.seconds/max(astra.calls,1)))


## Shut down

The task helper leaves the gripper empty first: a run that ends with the object held places it or
puts it back. Then it moves clear toward the base, and the arm folds and powers off.


In [ ]:
task.finish(runner, arm)   # leaves the gripper empty: places the object or puts it back
arm.close()
rec.close()
cam.close()
